In [3]:
!pip install pyspark

  Using cached pyspark-4.1.2-py2.py3-none-any.whl
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 'import': The Python command to bring in external code toolkits.
# 'os': The "Operating System" toolkit. We need this specifically to use 'os.environ' to create global system variables.
import os 

# 'sys': The "System Parameters" toolkit. We need this specifically to use 'sys.executable' to find the exact file path of our venv's python.exe.
import sys 

# 'from': Tells Python to look inside a specific folder/module rather than importing everything.
# 'pyspark.sql': The specific PySpark module that contains all the tools for DataFrames and SQL.
# 'import SparkSession': Brings in ONLY the 'SparkSession' class (the blueprint for our Manager), instead of the entire module.
from pyspark.sql import SparkSession
import pandas as pd


# --- FIX FOR WINDOWS VIRTUAL ENVIRONMENTS ---
# sys.executable finds the exact file path to our isolated venv Python.

# 1. THE WORKERS: Tells Spark's Java Manager exactly which Python to use 
# when actually doing the math and processing the data across your CPU cores.
os.environ["PYSPARK_PYTHON"] = sys.executable

# 2. THE MANAGER (DRIVER): Tells Spark exactly which Python to use 
# to read this notebook, understand your code, and plan out the tasks.
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


# 1. Initialize the Spark Engine
# We are building the "manager" that will handle our distributed data.
spark = (
    SparkSession.builder
    .appName("Grocery_ETL_Practice")
    .master("local[*]")
    .getOrCreate()
)

# or # Hard to read
# spark = SparkSession.builder.appName("Grocery_ETL_Practice").master("local[*]").getOrCreate()   
# .builder: Starts the setup process.
# .appName: Gives our application a name (useful when debugging later).
# .master("local[*]"): The magic command. It tells Spark to use ALL available CPU cores ("*") on your laptop to simulate a network cluster.
# .getOrCreate(): Starts the engine or connects to it if it's already running.

print("The Spark Engine is running and ready!")

ModuleNotFoundError: No module named 'pyspark'

In [2]:
# Create some raw mock data for an inventory system
raw_data = [
    ("Apples", "Fruit", 1.50, 100),
    ("Bananas", "Fruit", 0.50, 250),
    ("Broccoli", "Vegetable", 2.00, 50),
    ("Carrots", "Vegetable", 1.20, 120),
    ("Milk", "Dairy", 3.50, 40)
]

columns = ["item", "Category", "Price", "Quantity"]

# --- PANDAS APPROACH ---
# This forces the entire dataset into one single block of your computer's RAM.
pdf = pd.DataFrame(raw_data, columns=columns)
print("--- Pandas DataFrame ---")
print(pdf)
print(f"Pandas Shape (Logical): {pdf.shape}")

# --- PYSPARK APPROACH ---
# This takes the data, slices it into smaller partitions, and distributes it across the CPU cores we activated in Cell 1.
sdf = spark.createDataFrame(raw_data,columns)
print("--- PySpark DataFrame ---")
sdf.show() # .show() is how we print data in Spark
print(f"PySpark Partitions (Physical slices): {sdf.rdd.getNumPartitions()}")

--- Pandas DataFrame ---
       item   Category  Price  Quantity
0    Apples      Fruit    1.5       100
1   Bananas      Fruit    0.5       250
2  Broccoli  Vegetable    2.0        50
3   Carrots  Vegetable    1.2       120
4      Milk      Dairy    3.5        40
Pandas Shape (Logical): (5, 4)
--- PySpark DataFrame ---
+--------+---------+-----+--------+
|    item| Category|Price|Quantity|
+--------+---------+-----+--------+
|  Apples|    Fruit|  1.5|     100|
| Bananas|    Fruit|  0.5|     250|
|Broccoli|Vegetable|  2.0|      50|
| Carrots|Vegetable|  1.2|     120|
|    Milk|    Dairy|  3.5|      40|
+--------+---------+-----+--------+

PySpark Partitions (Physical slices): 12


In [3]:
# 'from': Tells Python to look inside a specific folder/module.
# 'pyspark.sql.functions': The PySpark folder that contains built-in math and data manipulation tools.
# 'import': Brings the specific tool we need into our current notebook.
# 'col': The specific tool that converts a text string (like "Category") into a Spark DataFrame column object so we can safely use it in our .filter() and .withColumn() recipes.
from pyspark.sql.functions import col

# 1. TRANSFORMATIONS (The "Lazy" steps)
# If you run this part without an Action at the end, Spark does ZERO math. 
# The Manager simply writes down these instructions on a notepad (creating an Execution Plan).
transformed_sdf = (
    sdf.filter(col("Category") == "Vegetable") # Keep only the vegetables
    .withColumn("Total_Value", col("Price") * col("Quantity")) # Multiply price by quantity to make a new column
)
                     
# 2. ACTIONS (The "Active" steps)
# Commands like .show(), .count(), or .write trigger the engine. 
# When Spark sees this line, the Manager finally looks at the notepad, 
# figures out the fastest way to do the math, and sends the work to your 12 CPU cores.
print("--- Final Transformed Data ---")
transformed_sdf.show()

--- Final Transformed Data ---
+--------+---------+-----+--------+-----------+
|    item| Category|Price|Quantity|Total_Value|
+--------+---------+-----+--------+-----------+
|Broccoli|Vegetable|  2.0|      50|      100.0|
| Carrots|Vegetable|  1.2|     120|      144.0|
+--------+---------+-----+--------+-----------+



### How Spark Thinks: The Physical Plan
When we use `.explain()`, Spark prints its internal "recipe" (read from bottom to top):
1. **Scan ExistingRDD:** Finds the raw data across the CPU cores.
2. **Filter:** Applies our `.filter()` rules (and safely ignores empty rows).
3. **Project:** Selects the columns and calculates the new math (`Price * Quantity`).
*Because of Lazy Evaluation, Spark figures out this perfect plan instantly without touching the data!*

In [9]:
transformed_sdf.explain()

== Physical Plan ==
*(1) Project [item#51, Category#52, Price#53, Quantity#54L, (Price#53 * cast(Quantity#54L as double)) AS Total_Value#68]
+- *(1) Filter (isnotnull(Category#52) AND (Category#52 = Vegetable))
   +- *(1) Scan ExistingRDD[item#51,Category#52,Price#53,Quantity#54L]




In [4]:
# We removed the .filter() so that Spark calculates the Total_Value for EVERY item.
transformed_sdf1 = (
    sdf.withColumn("Total_Value", col("Price") * col("Quantity")) 
)
                     
print("--- Transformed Data (All Categories) ---")
transformed_sdf1.show()

--- Transformed Data (All Categories) ---
+--------+---------+-----+--------+-----------+
|    item| Category|Price|Quantity|Total_Value|
+--------+---------+-----+--------+-----------+
|  Apples|    Fruit|  1.5|     100|      150.0|
| Bananas|    Fruit|  0.5|     250|      125.0|
|Broccoli|Vegetable|  2.0|      50|      100.0|
| Carrots|Vegetable|  1.2|     120|      144.0|
|    Milk|    Dairy|  3.5|      40|      140.0|
+--------+---------+-----+--------+-----------+



In [5]:
# 1. GROUPING AND AGGREGATING
# We tell Spark to put all identical categories into buckets, and then add up the total value in each bucket.
summary_sdf = transformed_sdf1.groupBy("Category").sum("Total_Value")

# 2. THE ACTION
# This triggers the Manager to create a new physical plan and send it to the workers.
print("--- Grand Total Inventory Value by Category ---")
summary_sdf.show()

--- Grand Total Inventory Value by Category ---
+---------+----------------+
| Category|sum(Total_Value)|
+---------+----------------+
|    Fruit|           275.0|
|Vegetable|           244.0|
|    Dairy|           140.0|
+---------+----------------+



In [15]:
# --- THE LOAD STEP (Saving to your hard drive) ---

# .write: Tells Spark we want to output data
# .mode("overwrite"): If we run this cell twice, it will replace the old file instead of crashing
# .option("header", "true"): Ensures our column names are saved at the top
# .csv(): The format we want to save it as, and the name of the destination
summary_sdf.write.mode("overwrite").option("header", "true").csv("inventory_summary_output")

print("Data successfully saved!")

Py4JJavaError: An error occurred while calling o174.csv.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(Ra

In [1]:
# Import SparkSession class.
# SparkSession is the entry point to work with Spark.
from pyspark.sql import SparkSession


# Create a SparkSession object.
# Think of this as starting the Spark Engine.
spark = (
    SparkSession.builder

    # Tell Spark where to run.
    # local[*] means:
    # local   -> Run on this computer.
    # *       -> Use all available CPU cores.
    .master("local[*]")

    # Give a name to your Spark application.
    # This name appears in Spark UI and logs.
    .appName("Employee Data Cleaning")

    # Create a new SparkSession.
    # If one already exists, reuse it.
    .getOrCreate()
)

ModuleNotFoundError: No module named 'pyspark'